In [ ]:
# Source Code 3
# Script to to threshold images in batches.

import os
import numpy as np
import cv2

# Define the input and output folder paths
input_folder = r'E:\wi\0-1000000'
output_folder = r'E:\wi\0-1000000-thresholded'

# Ensure the destination folder exists, create it if not
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

batch_size = 100
threshold_value = 128 # Adjust this threshold value as needed
kernel = np.ones((3, 3), np.uint8) # Adjust the kernel size as needed
white_range = (230, 245) # Adjust the white range as needed

file_names = [file_name for file_name in os.listdir(input_folder) if file_name.endswith(".tif")]
num_files = len(file_names)

for batch_start in range(0, num_files, batch_size):
    batch_end = min(batch_start + batch_size, num_files)
    batch_file_names = file_names[batch_start:batch_end]
    
    for file_name in batch_file_names:
        tif_path = os.path.join(input_folder, file_name)
        img = cv2.imread(tif_path, cv2.IMREAD_GRAYSCALE)
        
        unique_values = np.unique(img)
        if len(unique_values) == 1:
            print(f"Skipping {file_name} - All cell values identical.")
            continue
            
        mask = img < threshold_value
        img[mask] = 255
        dilation = cv2.dilate(img, kernel, iterations=3)
        output_filename = os.path.splitext(file_name)[0] + ".png"
        output_path = os.path.join(output_folder, output_filename)
        cv2.imwrite(output_path, dilation)

        processed_img = cv2.imread(output_path, cv2.IMREAD_GRAYSCALE)
        processed_img[~np.logical_and(processed_img >= white_range[0],
                                      processed_img <= white_range[1])] = 0
        processed_img[np.logical_and(processed_img >= white_range[0],
                                     processed_img <= white_range[1])] = 255
        cv2.imwrite(output_path, processed_img)